# NB9a - 5-fold cross-validation for the graph models (A3, A4, A5; A1 optional)

**Run on:** GPU (T4/P100), Internet ON. **Input to add (new versions only):** `eurosat-geograph-nb1-v2`, `eurosat-geograph-nb3-v2`, the NB2 output dataset (recipe in `nb2_config.json`).

Protocol (frozen after Phase 1): graph configuration K=25, m=20; recipe identical to NB2/NB4 (30 epochs, AdamW lr 3e-4, wd 1e-4, cosine, label smoothing 0.1, seed 42). Fold k: **test = fold k, validation = fold (k+1) mod 5, training = the other three**, normalisation statistics from the folds other than k (`channel_stats.json`).

Run a different `CV_FOLDS` in each notebook/session (for example [0,1,2] and [3,4]); every run saves `nb9a_cv_results_<tag>.csv` and `nb9a_cv_probs_<tag>.npz`. Merge them afterwards (NB8). **Quick test first:** `CV_FOLDS=[0]`, `CV_EXPERIMENTS=["A3"]`, `EPOCHS_CV=2`.


## Cell 1 - Configuration, GPU check, locate the data from NB1, NB3 (and NB2)

In [1]:
import os, glob, json, time, random, subprocess, sys, re
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True); import timm
try:
    import torch_geometric
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=True); import torch_geometric

WORK = "/kaggle/working"; os.makedirs(WORK, exist_ok=True)

# ---- recipe: identical to NB2 (copied from nb2_config.json when found) ----
CFG = dict(seed=42, epochs=30, batch_size=128, lr=3e-4, weight_decay=1e-4, label_smoothing=0.1, drop_rate=0.2,
           img_size=64, num_workers=2, backbone="tf_efficientnetv2_s.in21k_ft_in1k",
           D=128, heads=4, layers=2, k_select_epochs=15)
REQUIRE_NB2 = True      # set False only for a quick test without NB2
nb2cfg_path = sorted(glob.glob("/kaggle/input/**/nb2_config.json", recursive=True))
if nb2cfg_path:
    nb2cfg = json.load(open(nb2cfg_path[0]))
    for k in ("seed", "epochs", "batch_size", "lr", "weight_decay", "label_smoothing", "drop_rate", "img_size"):
        if k in nb2cfg: CFG[k] = nb2cfg[k]
    print("Recipe copied from NB2:", {k: CFG[k] for k in ("seed", "epochs", "batch_size", "lr", "weight_decay", "label_smoothing", "img_size")})
else:
    REQ = "NB2 output (nb2_config.json, nb2_probs.npz) is missing from Input: add the NEW NB2 output dataset (needed for the same recipe and McNemar vs B2)."
    if REQUIRE_NB2: raise AssertionError(REQ)
    print("WARNING:", REQ, "Using the default recipe.")

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
seed_all(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
      "| torch", torch.__version__, "| PyG", torch_geometric.__version__, "| timm", timm.__version__)

def find(pattern):
    m = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return m

x16 = find("X16_float16.npy"); assert x16, "NB1 output not found (X16_float16.npy)"
assert len(x16) == 1, f"Several X16_float16.npy in Input (old NB1 still attached?). Keep only the fixed NB1 dataset: {x16}"
NB1_DIR = os.path.dirname(x16[0])
gfiles = find("graphs_K*_m*.npz"); assert gfiles, "NB3 output not found (graphs_K*_m*.npz)"
GRAPH_FILES = {}
for f in gfiles:
    mm = re.search(r"graphs_K(\d+)_m(\d+)\.npz", os.path.basename(f))
    GRAPH_FILES[(int(mm.group(1)), int(mm.group(2)))] = f
assert len(GRAPH_FILES) == len(gfiles), f"Duplicate graph files in Input (old NB3 still attached?): {gfiles}"
print("NB1_DIR:", NB1_DIR); print("graph configs found:", sorted(GRAPH_FILES))

idx = pd.read_csv(f"{NB1_DIR}/folds.csv")
stats = json.load(open(f"{NB1_DIR}/channel_stats.json"))
y_all = np.load(f"{NB1_DIR}/labels.npy")
try:
    X16 = np.load(f"{NB1_DIR}/X16_float16.npy")
except MemoryError:
    X16 = np.load(f"{NB1_DIR}/X16_float16.npy", mmap_mode="r"); print("X16 loaded as memmap")
assert X16.shape == (len(idx), 16, 64, 64) and (idx["label"].values == y_all).all()
CLASSES = [idx.loc[idx["label"] == i, "class"].iloc[0] for i in range(10)]
FOLDS = idx["fold"].values
_ri = np.sort(np.random.default_rng(0).choice(len(X16), 2000, replace=False))
_ndbi = float(np.asarray(X16[_ri, 15], dtype=np.float32).mean())
print("NDBI mean (random 2000 patches): %.4f  (fixed NB1 = about -0.18; old B12-based = about -0.38)" % _ndbi)
assert _ndbi > -0.25, "OLD X16 detected (B12-based NDBI). Attach the fixed NB1 output only."
print("X16:", X16.shape, "| classes:", CLASSES)

Recipe copied from NB2: {'seed': 42, 'epochs': 30, 'batch_size': 128, 'lr': 0.0003, 'weight_decay': 0.0001, 'label_smoothing': 0.1, 'img_size': 64}
Device: cuda | GPU: Tesla T4 | torch 2.11.0+cu130 | PyG 2.8.1 | timm 1.0.30
NB1_DIR: /kaggle/input/datasets/sonalidas6555/eurosat-nb1-output
graph configs found: [(25, 20), (36, 20)]
NDBI mean (random 2000 patches): -0.1872  (fixed NB1 = about -0.18; old B12-based = about -0.38)
X16: (27000, 16, 64, 64) | classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


## Cell 2 - Model module (`geograph_model.py`)

The model, dataset, augmentation and collate code is written to a module so that NB6 (robustness) and NB7 (explainability) import exactly the same code.

In [2]:
MODEL_CODE = r'''
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import timm
from torch_geometric.nn import GCNConv, GATv2Conv, TransformerConv
from torch_geometric.utils import softmax as pyg_softmax

N_STATIC = 36
CX, CY = 32, 33        # centroid columns in the static node features (see NB3)

EXPERIMENTS = {
    "A1":  dict(kind="gcn",         edge_attr=False, spectral=False, fusion="concat"),
    "A2":  dict(kind="gatv2",       edge_attr=False, spectral=False, fusion="concat"),
    "A2b": dict(kind="gatv2",       edge_attr=True,  spectral=False, fusion="concat"),
    "A3":  dict(kind="transformer", edge_attr=True,  spectral=False, fusion="concat"),
    "A4":  dict(kind="transformer", edge_attr=True,  spectral=True,  fusion="concat"),
    "A5":  dict(kind="transformer", edge_attr=True,  spectral=True,  fusion="gated"),
}


# ---------------------------------------------------------------- numpy side: augmentation, statistics, dataset
def augment_geom(img, seg, node):
    # the same rotation / flips for the image, the segmentation map and the centroids (x, y in [0, 1])
    node = node.copy()
    for _ in range(random.randint(0, 3)):
        img = np.rot90(img, 1, axes=(1, 2)); seg = np.rot90(seg, 1)
        cx, cy = node[:, CX].copy(), node[:, CY].copy()
        node[:, CX], node[:, CY] = cy, 1.0 - cx
    if random.random() < 0.5:                                  # flip along the width axis
        img = np.flip(img, axis=2); seg = np.flip(seg, axis=1); node[:, CX] = 1.0 - node[:, CX]
    if random.random() < 0.5:                                  # flip along the height axis
        img = np.flip(img, axis=1); seg = np.flip(seg, axis=0); node[:, CY] = 1.0 - node[:, CY]
    return np.ascontiguousarray(img), np.ascontiguousarray(seg), node


def graph_stats(G, train_idx, use_spectral):
    # mean/std of node features and edge attributes over the TRAINING graphs only
    n_per, e_per = np.diff(G["node_ptr"]), np.diff(G["edge_ptr"])
    in_tr = np.isin(np.arange(len(n_per)), train_idx)
    nodes = G["node_static"][np.repeat(in_tr, n_per)]
    emask = np.repeat(in_tr, e_per)
    if not use_spectral:
        emask = emask & (G["edge_type"] == 0)
    ea = G["edge_attr"][emask]
    return (nodes.mean(0).astype(np.float32), (nodes.std(0) + 1e-6).astype(np.float32),
            ea.mean(0).astype(np.float32), (ea.std(0) + 1e-6).astype(np.float32))


class GraphDataset(Dataset):
    def __init__(self, indices, G, X16, y, img_mu, img_sd, nstat, use_spectral, augment):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.G, self.X16, self.y = G, X16, y
        self.mu = img_mu.astype(np.float32)[:, None, None]; self.sd = img_sd.astype(np.float32)[:, None, None]
        self.nmu, self.nsd, self.emu, self.esd = nstat
        self.use_spectral, self.augment = use_spectral, augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        r = int(self.indices[i]); G = self.G
        img = (np.asarray(self.X16[r], dtype=np.float32) - self.mu) / (self.sd + 1e-6)
        seg = G["seg"][r].astype(np.int64)
        a, b = int(G["node_ptr"][r]), int(G["node_ptr"][r + 1]); c, d = int(G["edge_ptr"][r]), int(G["edge_ptr"][r + 1])
        node = G["node_static"][a:b].copy()
        ei = G["edge_index"][:, c:d].astype(np.int64); ea = G["edge_attr"][c:d]
        if not self.use_spectral:
            keep = G["edge_type"][c:d] == 0
            ei, ea = ei[:, keep], ea[keep]
        if self.augment:
            img, seg, node = augment_geom(img, seg, node)
        node = (node - self.nmu) / self.nsd
        ea = (ea - self.emu) / self.esd
        return img, seg, node, ei, ea, int(self.y[r])


def collate_numpy(items):
    sizes = [len(it[2]) for it in items]
    offs = np.concatenate([[0], np.cumsum(sizes)]).astype(np.int64)
    return dict(
        x=np.stack([it[0] for it in items]).astype(np.float32),
        node_ids=np.stack([it[1] + offs[i] for i, it in enumerate(items)]).astype(np.int64),
        node_static=np.concatenate([it[2] for it in items]).astype(np.float32),
        edge_index=np.concatenate([it[3] + offs[i] for i, it in enumerate(items)], axis=1).astype(np.int64),
        edge_attr=np.concatenate([it[4] for it in items]).astype(np.float32),
        batch=np.repeat(np.arange(len(items)), sizes).astype(np.int64),
        y=np.array([it[5] for it in items], dtype=np.int64))


def collate(items):
    return {k: torch.from_numpy(v) for k, v in collate_numpy(items).items()}


# ---------------------------------------------------------------- torch side
def pool_regions(fmap, node_ids, n_nodes):
    # masked mean and max pooling of the dense map inside every superpixel (all graphs of the batch at once)
    f = fmap.float()
    D = f.size(1)
    f = f.permute(0, 2, 3, 1).reshape(-1, D)
    ids = node_ids.reshape(-1)
    s = f.new_zeros(n_nodes, D).index_add_(0, ids, f)
    cnt = torch.bincount(ids, minlength=n_nodes).clamp(min=1).unsqueeze(1).to(f.dtype)
    mx = f.new_full((n_nodes, D), float("-inf")).scatter_reduce_(0, ids.unsqueeze(1).expand(-1, D), f,
                                                                  reduce="amax", include_self=True)
    return torch.cat([s / cnt, mx], dim=1)


class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Linear(d, 1)

    def forward(self, h, batch, B):
        a = pyg_softmax(self.gate(h).squeeze(-1), batch, num_nodes=h.size(0))
        z = h.new_zeros(B, h.size(1)).index_add_(0, batch, h * a.unsqueeze(-1))
        return z, a


class GraphBlock(nn.Module):
    def __init__(self, kind, d, heads, edge_dim, drop):
        super().__init__()
        self.kind = kind
        if kind == "gcn":
            self.conv = GCNConv(d, d)
        elif kind == "gatv2":
            self.conv = GATv2Conv(d, d, heads=heads, concat=False, edge_dim=edge_dim, dropout=drop)
        else:
            self.conv = TransformerConv(d, d, heads=heads, concat=False, edge_dim=edge_dim, dropout=drop)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)

    def forward(self, h, ei, ea):
        out = self.conv(h, ei) if self.kind == "gcn" else self.conv(h, ei, ea)
        return self.norm(h + self.drop(F.elu(out)))


class GeoGraphNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg["D"]
        self.cfg, self.img, self.pretrained = cfg, cfg.get("img_size", 64), True
        kw = dict(features_only=True, out_indices=(1, 2, 3, 4), in_chans=16)
        try:
            self.stem = timm.create_model(cfg["backbone"], pretrained=True, **kw)
        except Exception as e:
            print("WARNING: pretrained weights unavailable (%s) -> training the stem from scratch" % type(e).__name__)
            self.stem = timm.create_model(cfg["backbone"], pretrained=False, **kw)
            self.pretrained = False
        chs = self.stem.feature_info.channels()
        self.proj = nn.ModuleList([nn.Conv2d(c, D, 1) for c in chs])
        self.dense_bn = nn.BatchNorm2d(D)
        self.cnn_vec = nn.Sequential(nn.Linear(chs[-1], D), nn.LayerNorm(D))
        self.node_in = nn.Sequential(nn.Linear(2 * D + N_STATIC, D), nn.LayerNorm(D), nn.ELU())
        edge_dim = 3 if cfg["edge_attr"] else None
        self.blocks = nn.ModuleList([GraphBlock(cfg["kind"], D, cfg["heads"], edge_dim, cfg["drop"])
                                     for _ in range(cfg["layers"])])
        self.pool = AttnPool(D)
        if cfg["fusion"] == "gated":
            self.gate = nn.Linear(2 * D, D)
        else:
            self.fuse = nn.Sequential(nn.Linear(2 * D, D), nn.ELU())
        self.drop = nn.Dropout(cfg["drop"])
        self.head = nn.Linear(D, 10)
        self.last_gate, self.last_attn = None, None

    def forward(self, b):
        x = b["x"]
        B = x.size(0)
        xin = x if self.img == 64 else F.interpolate(x, size=(self.img, self.img), mode="bilinear", align_corners=False)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=x.is_cuda):
            feats = self.stem(xin)
            dense = None
            for p, f in zip(self.proj, feats):
                u = F.interpolate(p(f), size=(64, 64), mode="bilinear", align_corners=False)
                dense = u if dense is None else dense + u
            dense = F.relu(self.dense_bn(dense))
            z_cnn = self.cnn_vec(feats[-1].mean(dim=(2, 3)))
        with torch.autocast(device_type="cuda", enabled=False):
            z_cnn = z_cnn.float()
            nodes = pool_regions(dense, b["node_ids"], b["node_static"].size(0))
            h = self.node_in(torch.cat([nodes, b["node_static"]], dim=1))
            ea = b["edge_attr"] if self.cfg["edge_attr"] else None
            for blk in self.blocks:
                h = blk(h, b["edge_index"], ea)
            z_gnn, attn = self.pool(h, b["batch"], B)
            both = torch.cat([z_cnn, z_gnn], dim=1)
            if self.cfg["fusion"] == "gated":
                g = torch.sigmoid(self.gate(both))
                z = g * z_cnn + (1.0 - g) * z_gnn
                self.last_gate = g.mean(dim=1).detach()
            else:
                z = self.fuse(both)
            self.last_attn = attn.detach()
            return self.head(self.drop(z))
'''
open("/kaggle/working/geograph_model.py", "w").write(MODEL_CODE)

import importlib
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")
import geograph_model as gm
importlib.reload(gm)
print("geograph_model.py written and imported | experiments:", list(gm.EXPERIMENTS))

geograph_model.py written and imported | experiments: ['A1', 'A2', 'A2b', 'A3', 'A4', 'A5']


## Cell 3 - Self-check of the geometry code (no GPU needed)

The augmentation must rotate/flip the image, the segmentation map and the node centroids in the same way.
The check recomputes the centroids from the transformed segmentation map and compares them with the transformed centroid features.

In [3]:
def _check_augmentation(n_trials=200):
    rng = np.random.default_rng(0)
    worst = 0.0
    for t in range(n_trials):
        seg = np.kron(rng.integers(0, 6, (4, 4)), np.ones((16, 16), int))
        seg = np.unique(seg, return_inverse=True)[1].reshape(64, 64) + 0           # random blocky map
        seg = np.unique(seg.ravel(), return_inverse=True)[1].reshape(64, 64)
        n = seg.max() + 1
        yy, xx = np.mgrid[0:64, 0:64]
        node = np.zeros((n, gm.N_STATIC), np.float32)
        for k in range(n):
            node[k, gm.CX] = xx[seg == k].mean() / 63.0; node[k, gm.CY] = yy[seg == k].mean() / 63.0
        img = np.broadcast_to(seg[None].astype(np.float32), (16, 64, 64)).copy()
        random.seed(t)
        img2, seg2, node2 = gm.augment_geom(img, seg, node)
        for k in range(n):
            cx = xx[seg2 == k].mean() / 63.0; cy = yy[seg2 == k].mean() / 63.0
            worst = max(worst, abs(cx - node2[k, gm.CX]), abs(cy - node2[k, gm.CY]))
        assert np.array_equal(img2[0], seg2.astype(np.float32)), "image and segmentation map are not transformed together"
    return worst

w = _check_augmentation()
print(f"augmentation check: worst centroid error {w:.2e}")
assert w < 1e-5, "centroids are not transformed consistently with the segmentation map"
print("OK: image, segmentation map and node centroids are augmented consistently")

augmentation check: worst centroid error 4.23e-08
OK: image, segmentation map and node centroids are augmented consistently


## Cell 4 - Splits, graph loading, loaders, metrics, trainer

Splits are identical to NB2 (Phase 1: test fold 0, validation fold 1, train folds 2-4; Phase 2 CV: test fold k, validation fold (k+1) mod 5).
`run_experiment` returns the test metrics (best-validation checkpoint), validation and test probabilities, the training history and, for A5, the mean gate value (weight of the CNN branch).

In [4]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, cohen_kappa_score, confusion_matrix)

def make_split(mode="phase1", k=0):
    if mode == "phase1":
        tr = np.where(np.isin(FOLDS, [2, 3, 4]))[0]; va = np.where(FOLDS == 1)[0]; te = np.where(FOLDS == 0)[0]
        st = stats["phase1_train_folds_2_3_4"]
        assert (idx["phase1"].values[tr] == "train").all() and (idx["phase1"].values[va] == "val").all() \
               and (idx["phase1"].values[te] == "test").all()
    else:
        te_f, va_f = k, (k + 1) % 5
        tr = np.where(~np.isin(FOLDS, [te_f, va_f]))[0]; va = np.where(FOLDS == va_f)[0]; te = np.where(FOLDS == te_f)[0]
        st = stats["cv_train_excluding_fold"][str(k)]
    assert not (set(tr) & set(te)) and not (set(va) & set(te)) and not (set(tr) & set(va))
    return dict(mode=mode, k=k, tr=tr, va=va, te=te, mu=np.array(st["mean"], np.float32), sd=np.array(st["std"], np.float32))

_GCACHE = {}
def load_graphs(gcfg):
    if gcfg not in _GCACHE:
        _GCACHE.clear()                                            # keep one graph set in memory
        _GCACHE[gcfg] = dict(np.load(GRAPH_FILES[gcfg]))
        G = _GCACHE[gcfg]; assert len(G["n_nodes"]) == len(idx), "graph file does not match folds.csv"
    return _GCACHE[gcfg]

def _winit(wid):
    s = (torch.initial_seed() + wid) % (2 ** 32); random.seed(s); np.random.seed(s)

def make_loaders(split, G, use_spectral):
    nstat = gm.graph_stats(G, split["tr"], use_spectral)
    bs, nw = CFG["batch_size"], CFG["num_workers"]
    g = torch.Generator(); g.manual_seed(CFG["seed"])
    kw = dict(num_workers=nw, pin_memory=True, worker_init_fn=_winit, persistent_workers=(nw > 0), collate_fn=gm.collate)
    mk = lambda ids, aug: gm.GraphDataset(ids, G, X16, y_all, split["mu"], split["sd"], nstat, use_spectral, aug)
    return (DataLoader(mk(split["tr"], True), batch_size=bs, shuffle=True, generator=g, **kw),
            DataLoader(mk(split["va"], False), batch_size=bs * 2, shuffle=False, **kw),
            DataLoader(mk(split["te"], False), batch_size=bs * 2, shuffle=False, **kw))

def compute_ece(probs, labels, n_bins=15):
    conf, pred = probs.max(axis=1), probs.argmax(axis=1)
    acc = (pred == labels).astype(np.float32); bins = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
    for b0, b1 in zip(bins[:-1], bins[1:]):
        m = (conf > b0) & (conf <= b1)
        if m.any(): ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def evaluate_predictions(probs, labels):
    p = probs.argmax(axis=1)
    return {"Acc": round(accuracy_score(labels, p) * 100, 3),
            "Macro_F1": round(f1_score(labels, p, average="macro", zero_division=0) * 100, 3),
            "Kappa": round(cohen_kappa_score(labels, p), 4), "ECE": round(compute_ece(probs, labels), 4)}

PAIRS = [("AnnualCrop", "PermanentCrop"), ("Residential", "Industrial"), ("HerbaceousVegetation", "Pasture"), ("River", "Highway")]
def class_pair_errors(cm, classes):
    tot = int(cm.sum() - np.trace(cm)); rows = []
    for a, b in PAIRS:
        i, j = classes.index(a), classes.index(b); t = int(cm[i, j] + cm[j, i])
        rows.append({"pair": f"{a} <-> {b}", "a_as_b": int(cm[i, j]), "b_as_a": int(cm[j, i]),
                     "total_confusions": t, "share_of_all_errors_pct": round(100.0 * t / max(tot, 1), 2)})
    return pd.DataFrame(rows)

def to_dev(b):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}

@torch.no_grad()
def predict_loader(model, loader):
    model.eval(); P, Y, Gt = [], [], []
    for b in loader:
        b = to_dev(b)
        logits = model(b)
        P.append(torch.softmax(logits.float(), -1).cpu().numpy()); Y.append(b["y"].cpu().numpy())
        if model.last_gate is not None: Gt.append(model.last_gate.cpu().numpy())
    return np.concatenate(P), np.concatenate(Y), (np.concatenate(Gt) if Gt else None)

@torch.no_grad()
def profile_model(model, loader, reps=10):
    model.eval()
    b = to_dev(next(iter(loader)))
    n = b["x"].size(0)
    gflops = float("nan")
    try:
        from torch.utils.flop_counter import FlopCounterMode
        b1 = to_dev(gm.collate([loader.dataset[0]]))
        with FlopCounterMode(display=False) as fc: model(b1)
        gflops = fc.get_total_flops() / 1e9
    except Exception as e:
        print("FLOP counting unavailable:", type(e).__name__)
    for _ in range(3): model(b)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t = time.time()
    for _ in range(reps): model(b)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return gflops, (time.time() - t) * 1000.0 / (n * reps)

def make_scaler(enabled):
    try: return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError): return torch.cuda.amp.GradScaler(enabled=enabled)

def run_experiment(code, split, gcfg, epochs=None, tag=""):
    epochs = epochs or CFG["epochs"]
    spec = gm.EXPERIMENTS[code]
    seed_all(CFG["seed"])
    G = load_graphs(gcfg)
    tr_dl, va_dl, te_dl = make_loaders(split, G, spec["spectral"])
    mcfg = dict(spec, D=CFG["D"], heads=CFG["heads"], layers=CFG["layers"], drop=CFG["drop_rate"],
                backbone=CFG["backbone"], img_size=CFG["img_size"])
    model = gm.GeoGraphNet(mcfg).to(DEVICE)
    params_m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    crit = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    use_amp = DEVICE.type == "cuda"; scaler = make_scaler(use_amp)
    name = f"{code}_K{gcfg[0]}m{gcfg[1]}{tag}"
    ckpt = f"{WORK}/best_{name}.pt"
    best, best_ep, hist, t0 = -1.0, 0, [], time.time()
    for ep in range(1, epochs + 1):
        model.train(); tl = 0.0
        for b in tr_dl:
            b = to_dev(b)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(b), b["y"])                          # autocast is handled inside the model (CNN part only)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(b["y"])
        sched.step()
        vp, vy, _ = predict_loader(model, va_dl); vres = evaluate_predictions(vp, vy)
        hist.append({"epoch": ep, "train_loss": tl / len(tr_dl.dataset), "val_acc": vres["Acc"], "val_macro_f1": vres["Macro_F1"], "val_ece": vres["ECE"]})
        if vres["Macro_F1"] > best:
            best, best_ep = vres["Macro_F1"], ep; torch.save(model.state_dict(), ckpt)
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"[{name}] Ep {ep:02d}/{epochs} | loss {tl/len(tr_dl.dataset):.4f} | val F1 {vres['Macro_F1']:.2f}% | best {best:.2f}% (ep {best_ep}) | {(time.time()-t0)/60:.1f} min")
    train_min = (time.time() - t0) / 60.0
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    te_p, te_y, gate = predict_loader(model, te_dl)               # test fold used once
    va_p, va_y, _ = predict_loader(model, va_dl)
    res = evaluate_predictions(te_p, te_y)
    gflops, ms = profile_model(model, te_dl)
    res.update({"Exp": code, "K": gcfg[0], "m": gcfg[1], "layer": spec["kind"], "edge_attr": spec["edge_attr"],
                "spectral_edges": spec["spectral"], "fusion": spec["fusion"], "Params_M": round(params_m, 2),
                "GFLOPs_approx": round(gflops, 3), "Infer_ms": round(ms, 4), "Train_Min": round(train_min, 2),
                "Best_Epoch": best_ep, "Best_Val_F1": best, "Gate_mean_CNN": (round(float(gate.mean()), 4) if gate is not None else None),
                "Pretrained": model.pretrained})
    print(f"--> {name} TEST: Acc={res['Acc']}% | Macro-F1={res['Macro_F1']}% | Kappa={res['Kappa']} | ECE={res['ECE']}\n")
    del model, opt
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res, te_p, te_y, va_p, va_y, pd.DataFrame(hist)

SPLIT = make_split("phase1")
print({k: len(SPLIT[k]) for k in ("tr", "va", "te")}, "| Phase 1 split OK")

{'tr': 16200, 'va': 5400, 'te': 5400} | Phase 1 split OK


## Cell 5 - Run the cross-validation folds (set the folds for this session)


In [5]:
BEST_GRAPH = (25, 20)
assert BEST_GRAPH in GRAPH_FILES, f"graph config {BEST_GRAPH} not found: {sorted(GRAPH_FILES)}"

CV_FOLDS = [0, 1, 2]                      # this session; use [3, 4] in the second session
CV_EXPERIMENTS = ["A3", "A4", "A5"]       # add "A1" if time allows
EPOCHS_CV = CFG["epochs"]                 # keep 30 (same as Phase 1); use 2 only for the quick test
TAG = "f" + "".join(str(k) for k in CV_FOLDS)

cv_rows, cv_probs = [], {}
for k in CV_FOLDS:
    sp = make_split("cv", k)
    print(f"=== fold {k}: train {len(sp['tr'])} | val {len(sp['va'])} | test {len(sp['te'])} ===")
    for code in CV_EXPERIMENTS:
        res, tp, ty, vp, vy, h = run_experiment(code, sp, BEST_GRAPH, epochs=EPOCHS_CV, tag=f"_f{k}")
        res["Fold"] = k; cv_rows.append(res)
        cv_probs[f"test_{code}_f{k}"] = tp; cv_probs[f"val_{code}_f{k}"] = vp; cv_probs[f"y_test_f{k}"] = ty; cv_probs[f"y_val_f{k}"] = vy
        pd.DataFrame(cv_rows).to_csv(f"{WORK}/nb9a_cv_results_{TAG}.csv", index=False)          # saved after every run
        np.savez_compressed(f"{WORK}/nb9a_cv_probs_{TAG}.npz", **cv_probs)
        p = f"{WORK}/best_{code}_K{BEST_GRAPH[0]}m{BEST_GRAPH[1]}_f{k}.pt"
        if os.path.exists(p): os.remove(p)                                                       # checkpoints are not needed afterwards
cv_df = pd.DataFrame(cv_rows)
display(cv_df[["Exp", "Fold", "Acc", "Macro_F1", "Kappa", "ECE", "Best_Epoch", "Train_Min"]])
display(cv_df.groupby("Exp")[["Acc", "Macro_F1", "Kappa", "ECE"]].agg(["mean", "std"]).round(3))


=== fold 0: train 16200 | val 5400 | test 5400 ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 86.5MB            

model.safetensors: downloading bytes:           |  0.00B            

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


[A3_K25m20_f0] Ep 01/30 | loss 0.7612 | val F1 95.83% | best 95.83% (ep 1) | 1.3 min
[A3_K25m20_f0] Ep 05/30 | loss 0.5392 | val F1 97.79% | best 97.79% (ep 5) | 6.3 min
[A3_K25m20_f0] Ep 10/30 | loss 0.5202 | val F1 97.47% | best 98.16% (ep 9) | 12.4 min
[A3_K25m20_f0] Ep 15/30 | loss 0.5125 | val F1 98.30% | best 98.45% (ep 11) | 18.6 min
[A3_K25m20_f0] Ep 20/30 | loss 0.5058 | val F1 98.56% | best 98.56% (ep 20) | 24.7 min
[A3_K25m20_f0] Ep 25/30 | loss 0.5052 | val F1 98.68% | best 98.68% (ep 25) | 30.9 min
[A3_K25m20_f0] Ep 30/30 | loss 0.5049 | val F1 98.70% | best 98.70% (ep 30) | 37.1 min
--> A3_K25m20_f0 TEST: Acc=98.889% | Macro-F1=98.828% | Kappa=0.9876 | ECE=0.0764



Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


... (13 outputs left out)

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


[A5_K25m20_f2] Ep 01/30 | loss 0.7384 | val F1 96.95% | best 96.95% (ep 1) | 1.3 min
[A5_K25m20_f2] Ep 05/30 | loss 0.5345 | val F1 98.11% | best 98.11% (ep 5) | 6.3 min
[A5_K25m20_f2] Ep 10/30 | loss 0.5174 | val F1 98.42% | best 98.42% (ep 10) | 12.5 min
[A5_K25m20_f2] Ep 15/30 | loss 0.5107 | val F1 98.66% | best 98.66% (ep 15) | 18.8 min
[A5_K25m20_f2] Ep 20/30 | loss 0.5069 | val F1 98.68% | best 98.72% (ep 19) | 25.0 min
[A5_K25m20_f2] Ep 25/30 | loss 0.5054 | val F1 98.57% | best 98.82% (ep 21) | 31.3 min
[A5_K25m20_f2] Ep 30/30 | loss 0.5050 | val F1 98.69% | best 98.82% (ep 21) | 37.5 min
--> A5_K25m20_f2 TEST: Acc=98.926% | Macro-F1=98.87% | Kappa=0.988 | ECE=0.0736



,Exp,Fold,Acc,Macro_F1,Kappa,ECE,Best_Epoch,Train_Min
0,A3,0,98.889,98.828,0.9876,0.0764,30,37.07
1,A4,0,98.944,98.886,0.9883,0.0785,26,37.49
2,A5,0,98.796,98.735,0.9866,0.0748,30,37.47
3,A3,1,98.685,98.618,0.9854,0.0764,25,37.02
4,A4,1,98.759,98.703,0.9862,0.0757,29,37.50
5,A5,1,98.889,98.828,0.9876,0.0743,23,37.48
6,A3,2,98.963,98.914,0.9885,0.0768,28,37.02
7,A4,2,99.019,98.968,0.9891,0.0754,20,37.48
8,A5,2,98.926,98.870,0.9880,0.0736,21,37.49


Acc        Macro_F1         Kappa           ECE       
       mean    std     mean    std   mean    std   mean    std
Exp                                                           
A3   98.846  0.144   98.787  0.152  0.987  0.002  0.077  0.000
A4   98.907  0.134   98.852  0.136  0.988  0.001  0.077  0.002
A5   98.870  0.067   98.811  0.069  0.987  0.001  0.074  0.001

## Cell 6 - List the outputs


In [6]:
print("files in", WORK)
for f in sorted(os.listdir(WORK)): print(f"  {f:40s} {os.path.getsize(os.path.join(WORK, f))/1e6:8.2f} MB")


files in /kaggle/working
  __notebook__.ipynb                           0.09 MB
  __pycache__                                  0.00 MB
  geograph_model.py                            0.01 MB
  nb9a_cv_probs_f012.npz                       3.33 MB
  nb9a_cv_results_f012.csv                     0.00 MB
